In [13]:
# LOAD
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
import faiss
import numpy as np


chunks = np.load("Data/chunks.npy", allow_pickle=True)
index = faiss.read_index("Data/index.faiss")

embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12532.91it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10003.91it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
# RETRIEVAL 
# retrieve top 20 chunks with the embedding model -> choose top 5 with the reranker model

def retrieve(query, k_initial=20, k_final=5):
    # Pasul A: Retrieval inițial (extrage mai multe fragmente)
    query_prefixed = "query: " + query
    q_emb = embedding_model.encode([query_prefixed])
    distances, indices = index.search(np.array(q_emb), k_initial)
    
    initial_chunks = [chunks[i] for i in indices[0]]
    
    # Pasul B: Reranking (Cross-Encoding)
    # Modelul compară perechi (întrebare, fragment) și dă un scor de relevanță
    sentence_combinations = [[query, chunk] for chunk in initial_chunks]
    rerank_scores = reranker_model.predict(sentence_combinations)
    
    # Pasul C: Sortăm fragmentele în funcție de scorul Reranker-ului
    reranked_results = sorted(
        list(zip(initial_chunks, rerank_scores)), 
        key=lambda x: x[1], 
        reverse=True
    )
    
    # Returnăm doar primele k_final cele mai bune
    return [res[0] for res in reranked_results[:k_final]]

QUESTION ANSWERING LLM

In [ ]:
# QUESTION ANSWERING LLM
import requests

def ask_llm(question, context_chunks):
    context = "\n\n--- FRAGMENT NOU ---\n\n".join(context_chunks)

    prompt = f"""
Ești un asistent virtual oficial care ajută candidații cu informații despre admiterea la FMI UBB Cluj.
Trebuie să fii clar, concis și să răspunzi DOAR pe baza contextului oferit.

REGULI STRICTE:
1. Răspunde EXCLUSIV în limba română.
2. Folosește DOAR informațiile din secțiunea "Context". Nu folosi cunoștințele tale generale.
3. Dacă răspunsul nu se află în "Context", nu încerca să ghicești. Scrie EXACT următoarea propoziție: "Nu dețin această informație conform datelor extrase de pe site-ul oficial."
4. Nu inventa formule de calcul, date sau taxe.

Context:
{context}

Întrebare:
{question}

Răspuns:
"""

    response = requests.post(
        "http://localhost:1234/v1/chat/completions",
        json={
            "model": "mistralai/mistral-7b-instruct-v0.3",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1
        }
    )


    data = response.json()
    
    if "choices" not in data:
        print("Eroare primită de la LM Studio:", data)
        return "Eroare la generarea răspunsului."

    return data["choices"][0]["message"]["content"]

SCORING LLM

In [29]:
# SCORING SYSTEM LLM
def score_llm(question, context_chunks, answer):
    context = "\n\n--- FRAGMENT NOU ---\n\n".join(context_chunks)

    prompt = f"""
### SYSTEM INSTRUCTION
You are a ruthless and objective evaluator of a RAG chatbot for university admissions.
Your task is to evaluate the chatbot's answer based STRICTLY on the provided Reference Context.
You MUST output ONLY a valid JSON object. Do not include any introductory text, concluding remarks, or explanations outside of the JSON.

### SCORING RUBRIC (Be extremely strict!)
1. Factual correctness (0-10): 10 = Perfect; 5 = Minor errors; 0 = Contradictions/Inventions.
2. Completeness (0-10): Did it fully answer based ONLY on the context?
3. Clarity (0-10): Romanian only. Score 0 if in English.
4. Hallucinations penalty (0-10): 10 = No hallucinations; 0 = Invented rules/data.

### CRITICAL RULE FOR "I DON'T KNOW"
If Answer == "Nu dețin această informație conform datelor extrase de pe site-ul oficial." AND information is missing from Context, score 10/10/10.

### EXAMPLE OUTPUT FORMAT
{{
  "brief_reason": "The answer correctly identifies the deadline but hallucinated the room number which was not in the context.",
  "factual": 7,
  "complete": 10,
  "clarity": 10,
  "no_hallucination": 5,
  "final_score": 8.0
}}

### EVALUATION DATA
Question: {question}
Chatbot Answer: {answer}
Reference Context: {context}

### FINAL MANDATE
Return ONLY the JSON object. Do not provide a "breakdown" or "note" after the JSON. Start your response with "{" and end it with "}".
"""

    response = requests.post(
        "http://localhost:1234/v1/chat/completions",
        json={
            "model": "meta-llama-3-8b-instruct",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0
        }
    )


    data = response.json()
    
    if "choices" not in data:
        print("Eroare primită de la LM Studio:", data)
        return "Eroare la generarea răspunsului."

    return data["choices"][0]["message"]["content"]

In [5]:
# QA Model function
def answer_question(question, relevant_chunks):
    answer = ask_llm(question, relevant_chunks)
    return answer

# LLM-as-a-Judge Model function
def score_answer(question, relevant_chunks, answer):
    answer = score_llm(question, relevant_chunks, answer)
    return answer

Model answers question & Judge scores them

In [6]:
import time
import json
import pandas as pd


def load_questions():
    with open("Data/questions.json", "r", encoding="utf-8") as f:
        return json.load(f)


def save_results(path, answers, scores):
    df = pd.DataFrame(answers)
    df.to_csv("Experiments/" + path + "/answers.csv", index=False, encoding="utf-8")

    df = pd.DataFrame(scores)
    df.to_csv("Experiments/" + path + "/scores.csv", index=False, encoding="utf-8")


def answerAndScore(path=None, initialK=20, finalK=5, question=None):
    questions = load_questions()
    answers = []
    scores = []
    totalTime = 0
    
    ct = 1
    if path != None and question == None:
        for q in questions:
            start = time.time()
            relevant_chunks = retrieve(q["question"], initialK, finalK)

            answer = answer_question(q["question"], relevant_chunks)
            print("Answered question ", ct) 
            answers.append({
                "id": q["id"],
                "question": q["question"],
                "category": q["category"],
                "difficulty": q["difficulty"],
                "answer": answer
            })

            score = score_answer(q, relevant_chunks, answer)
            print("Scored answer ", ct) ; ct+=1
            scores.append({
                "id": q["id"],
                "question": q["question"],
                "category": q["category"],
                "difficulty": q["difficulty"],
                "answer": answer,
                "score": score
            })

            elapsed = int(time.time() - start) ; minutes = elapsed // 60 ; seconds = elapsed % 60
            print(f"Time: {minutes}m{seconds}s")
            totalTime += elapsed

            save_results(path, answers, scores)
            
        print(f"Total time: {totalTime//60}m{totalTime%60}s")


    elif path == None and question != None:
        relevant_chunks = retrieve(question, initialK, finalK)
        answer = answer_question(question, relevant_chunks)
        print(answer) 
        score = score_answer(question, relevant_chunks, answer)
        print("\n\n",score) ; ct+=1
    else:
        print("Wrong parameter input")

Analyse scores

In [2]:
import pandas as pd
import json


def parse_score(text):
    text = str(text).strip()

    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()

    return json.loads(text)


def analyse_scores(path, file="scores.csv"):
    df = pd.read_csv("Experiments/" + path + "/" + file)

    df["score_dict"] = df["score"].apply(parse_score)

    df["final_score"] = df["score_dict"].apply(lambda x: x["final_score"])
    df["factual"] = df["score_dict"].apply(lambda x: x["factual"])
    df["complete"] = df["score_dict"].apply(lambda x: x["complete"])
    df["clarity"] = df["score_dict"].apply(lambda x: x["clarity"])

    print("Average final:", df["final_score"].mean())
    print("Average factual:", df["factual"].mean())
    print()
    print(df.groupby("difficulty")["final_score"].mean())
    print(df.groupby("category")["final_score"].mean())

Experiments

In [3]:
#answerAndScore("0_TwoQuestions_Test")
#Total: 2m58s

analyse_scores("0_TwoQuestions_Test")

Average final: 8.5
Average factual: 8.5

difficulty
easy    8.5
Name: final_score, dtype: float64
category
calendar_admitere     8.5
documente_necesare    8.5
Name: final_score, dtype: float64


In [ ]:
#answerAndScore("1_20Questions_Baseline")
# Total: 26m30s
analyse_scores("1_20Questions_Baseline")

# ANALIZA CRITICA:
# LLM-as-a-Judge a facut greseli / a fost prea indulgent.
# Pe baza raspunsurilor date de modelul QA, media finala ar fi trebuit sa fie ~5-6.

# Exemplu greseala:
# ID:10
# intrebare: "Admiterea se face pe bază de examen sau de dosar?"
# raspuns: "Admiterea se face pe bază de dosar, ..."  --> gresit

# Solutie propusa:
# Schimbare prompt LLM-as-a-Judge
# 

Average final: 8.361
Average factual: 8.65

difficulty
easy      8.591250
medium    8.138333
tricky    8.276667
Name: final_score, dtype: float64
category
calendar_admitere     8.800
cazuri_edge_case      8.415
contestatii           7.500
documente_necesare    8.500
inscriere_online      7.330
locuri_buget          7.915
medii_minime          8.000
probe                 8.500
specializari          8.750
taxe                  8.500
Name: final_score, dtype: float64


In [ ]:
#answerAndScore("2_newEmbeddingModel")
# Total time: 9m29s -> significantly faster

analyse_scores("2_newEmbeddingModel")

# Answers and scores are still bad
#  -> halucineaza
#  -> raspunde cateodata pe engleza
#  -> judge-ul este prea generos cu notele

Average final: 8.082
Average factual: 8.1

difficulty
easy      8.070000
medium    8.301667
tricky    7.878333
Name: final_score, dtype: float64
category
calendar_admitere     8.686667
cazuri_edge_case      7.865000
contestatii           9.060000
documente_necesare    8.500000
inscriere_online      8.500000
locuri_buget          8.000000
medii_minime          7.625000
probe                 8.500000
specializari          7.265000
taxe                  9.250000
Name: final_score, dtype: float64


In [ ]:
#answerAndScore("3_newPrompts")
# Total: 9m56s

analyse_scores("3_newPrompts")
# + Judge mai sever
# + QA Chatbot nu mai raspunde pe engleza & recunoaste lipsa de context
# - Judge se contrazice singur (question ID 9: "ar trebui sa fie asta 
#   in loc de asta" dar a zis exact aceeasi chestie)
# - Judge considera si "nu stiu" ca raspuns gresit (lipsa context = nu stiu = corect)
# - QA Chatbot mai face greseli (a amestecat o regulă despre cetățenie dublă cu o 
#   regulă despre master și identitate culturală. Cauza suspectata: chunk overlapping repeta chestii. Modelul crede
#   ca textul e continuarea la aceeasi categorie cand de fapt inceputul chunkului o repeta pe cea de inainte)

Average final: 7.791500000000001
Average factual: 7.1

difficulty
easy      8.037500
medium    8.404167
tricky    6.850833
Name: final_score, dtype: float64
category
calendar_admitere     10.00000
cazuri_edge_case       7.77625
contestatii           10.00000
documente_necesare     7.30000
inscriere_online       8.50000
locuri_buget           8.65000
medii_minime           3.87500
probe                  6.87500
specializari           7.12500
taxe                   8.50000
Name: final_score, dtype: float64


In [ ]:
#answerAndScore("4_promptsAndDetails")
# Total: 8m43s

analyse_scores("4_promptsAndDetails")
# context seems to be insufficient for the QA model

Average final: 7.6275
Average factual: 6.75

difficulty
easy      7.306250
medium    7.108333
tricky    8.575000
Name: final_score, dtype: float64
category
calendar_admitere     6.216667
cazuri_edge_case      8.300000
contestatii           4.700000
documente_necesare    3.750000
inscriere_online      8.500000
locuri_buget          8.250000
medii_minime          9.125000
probe                 7.250000
specializari          8.125000
taxe                  9.250000
Name: final_score, dtype: float64


In [ ]:
#answerAndScore("5_improvedRetrieval")
# Time: 8m55s

analyse_scores("5_improvedRetrieval")
# Topic Drift: ID11 - a gasit un cuvant ("repartizare"); credea ca e relevant, dar era caz particular pentru modulul pedagogic
#                     (nu a inteles contextul, a gasit doar un cuvant bun d.p.d.v. semantic)
# ID2 - QA model raspunde corect (24 de ore înainte).
#       LLM-as-a-Judge: „E corect factual, dar nu e în context, deci e halucinație” -> Scor 3.75
#       Interpretare: Informația nu se află în cele 5 fragmente pe care le primește modelul QA, 
#       dar modelul o știe din antrenamentul lui (cunoștințe generale).
#       !! Dacă universitatea schimbă regula la „48 de ore”, modelul va continua să zică „24” din amintiri.
       

Average final: 7.890000000000001
Average factual: 7.2

difficulty
easy      8.181250
medium    6.608333
tricky    8.783333
Name: final_score, dtype: float64
category
calendar_admitere     6.833333
cazuri_edge_case      8.625000
contestatii           8.500000
documente_necesare    9.250000
inscriere_online      8.500000
locuri_buget          7.100000
medii_minime          9.100000
probe                 6.250000
specializari          7.350000
taxe                  8.500000
Name: final_score, dtype: float64


In [ ]:
#answerAndScore("6_changedCrawler")
# Time: 9m4s

analyse_scores("6_changedCrawler")

Average final: 8.0425
Average factual: 7.05

difficulty
easy      7.968750
medium    7.516667
tricky    8.666667
Name: final_score, dtype: float64
category
calendar_admitere     7.483333
cazuri_edge_case      8.187500
contestatii           8.500000
documente_necesare    8.000000
inscriere_online      8.250000
locuri_buget          8.400000
medii_minime          8.600000
probe                 7.200000
specializari          8.175000
taxe                  7.000000
Name: final_score, dtype: float64


In [ ]:
#answerAndScore("7_changedModels")

analyse_scores("7_changedModels", file="scores4.csv")

Average final: 8.5375
Average factual: 8.0

difficulty
easy      8.937500
medium    8.458333
tricky    8.083333
Name: final_score, dtype: float64
category
calendar_admitere     10.0000
cazuri_edge_case       7.1250
contestatii           10.0000
documente_necesare     7.0000
inscriere_online      10.0000
locuri_buget          10.0000
medii_minime           6.2500
probe                  8.2500
specializari           8.9375
taxe                   8.7500
Name: final_score, dtype: float64


In [ ]:
# answerAndScore("8_changedCrawlerAndChunking")
# Total time: 3m49s

analyse_scores("8_changedCrawlerAndChunking")

Average final: 9.59
Average factual: 8.3

difficulty
easy      12.175000
medium     8.300000
tricky     7.433333
Name: final_score, dtype: float64
category
calendar_admitere      7.933333
cazuri_edge_case       8.200000
contestatii            7.500000
documente_necesare    40.000000
inscriere_online       9.000000
locuri_buget           8.750000
medii_minime           6.000000
probe                  9.000000
specializari           7.900000
taxe                   8.600000
Name: final_score, dtype: float64
